# Public InstanSeg marker-set comparison on SLIDE-0330

This notebook compares the public `fluorescence_nuclei_and_cells` v0.1.1 model with two channel selections on the exact same SLIDE-0330 crops:

- **Current markers:** the 11-channel set used by the current visual comparison.
- **Previous pipeline markers:** the 9-channel set read from the previous SLIDE-0330 `latest_instanseg.json` run record.

Everything else is held fixed: crop coordinates, source image, physical scale, public checkpoint, normalization, postprocessing, and native nucleus/cell resolution. This is a marker-selection ablation, not a replay of every historical pipeline setting. The old run record's settings are printed for reference.

In [ ]:
import gc
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import tifffile
import torch
import zarr
from skimage.segmentation import find_boundaries

SLIDE_ID = 'SLIDE-0330'
FULL_MERGE_OME = Path(
    '/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/'
    'SLIDE-0330/outputs_v3/SLIDE-0330_full_merge.ome.tif'
)
TRAINING_ROOT = Path(os.environ.get(
    'INSTANSEG_TRAINING_ROOT', '/data1/lowes/ratnayn/Data/instanseg'
)).expanduser().resolve()
SOURCE_ROOT = Path(os.environ.get(
    'INSTANSEG_EVAL_SOURCE_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg/slurm_runs/'
    'instanseg_multihead_0325_20260825/source/instanseg',
)).expanduser().resolve()
PUBLIC_MODEL_CACHE_ROOT = Path(os.environ.get(
    'INSTANSEG_PUBLIC_MODEL_CACHE',
    str(TRAINING_ROOT / 'public_model_cache'),
)).expanduser().resolve()
PREVIOUS_RUN_RECORD = Path(os.environ.get(
    'INSTANSEG_SLIDE0330_PREVIOUS_RUN_RECORD',
    '/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/'
    'SLIDE-0330/outputs_v3/run_records/latest_instanseg.json',
)).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get(
    'INSTANSEG_MARKER_SET_RESULTS_ROOT',
    '/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/'
    'slide0330_public_marker_set_comparison',
)).expanduser().resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = os.environ.get('INSTANSEG_VISUALIZATION_DEVICE', 'cuda:0')
if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA notebook kernel before running model inference.')

CROP_SIZE_PX = 768
N_CROPS = 6
RANDOM_SEED = 3300330
RESOLVE_CELL_AND_NUCLEUS = True
SAVE_FIGURES = True

CURRENT_MARKERS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_PODOPLANIN_750', 'R8_F480_D2S9R_555',
    'R9_CD68_E3O7V_488', 'R12_CD3E_E4T1B_AF555',
]
previous_run = json.loads(PREVIOUS_RUN_RECORD.read_text())
OLD_SLIDE_CONFIG = previous_run['resolved_slide_config']
PREVIOUS_PIPELINE_MARKERS = OLD_SLIDE_CONFIG['instanseg']['channels']
OLD_INSTANSEG_CONFIG = OLD_SLIDE_CONFIG['instanseg']

# These are held fixed for both marker sets. They are the public-model settings
# used by the current small-image comparison, rather than the old run's
# seed_threshold/tile_size settings.
POSTPROCESSING = {
    'min_size': 10,
    'mask_threshold': 0.53,
    'peak_distance': 5,
    'seed_threshold': 0.7,
    'overlap_threshold': 0.3,
    'mean_threshold': 0.0,
    'fg_threshold': 0.5,
    'window_size': 32,
    'cleanup_fragments': True,
    'resolve_cell_and_nucleus': RESOLVE_CELL_AND_NUCLEUS,
}

MARKER_SET_SPECS = {
    'current': {'label': 'Current 11-channel set', 'channels': CURRENT_MARKERS},
    'previous_pipeline': {
        'label': 'Previous pipeline 9-channel set',
        'channels': PREVIOUS_PIPELINE_MARKERS,
    },
}
if set(CURRENT_MARKERS) == set(PREVIOUS_PIPELINE_MARKERS):
    raise RuntimeError('The two marker sets are unexpectedly identical.')
print({'slide': SLIDE_ID, 'device': DEVICE, 'resolve': RESOLVE_CELL_AND_NUCLEUS})
display(pd.DataFrame({
    'current_only': pd.Series(sorted(set(CURRENT_MARKERS) - set(PREVIOUS_PIPELINE_MARKERS))),
    'previous_only': pd.Series(sorted(set(PREVIOUS_PIPELINE_MARKERS) - set(CURRENT_MARKERS))),
}))
print('Previous run record:', PREVIOUS_RUN_RECORD)
print('Previous run record SHA256:', hashlib.sha256(PREVIOUS_RUN_RECORD.read_bytes()).hexdigest())
display(pd.DataFrame([
    {'setting': key, 'previous_pipeline_value': (
        OLD_SLIDE_CONFIG.get(key) if key == 'pixel_size_um' else OLD_INSTANSEG_CONFIG.get(key)
    )}
    for key in ['model', 'mode', 'tile_size', 'batch_size', 'resolve_cell_and_nucleus',
                 'cleanup_fragments', 'seed_threshold', 'pixel_size_um']
]))

## Load the public v0.1.1 model through the native API

In [ ]:
if not FULL_MERGE_OME.is_file():
    raise FileNotFoundError(FULL_MERGE_OME)
if not PREVIOUS_RUN_RECORD.is_file():
    raise FileNotFoundError(PREVIOUS_RUN_RECORD)
if not (SOURCE_ROOT / 'instanseg').is_dir():
    raise FileNotFoundError(SOURCE_ROOT / 'instanseg')
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import instanseg
from instanseg import InstanSeg as NativeInstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as instanseg_inference_class
instanseg_inference_class.TiffSlide = TiffSlide

model_index_path = SOURCE_ROOT / 'instanseg' / 'bioimageio_models' / 'model-index.json'
entries = [
    entry for entry in json.loads(model_index_path.read_text())
    if entry.get('name') == 'fluorescence_nuclei_and_cells'
]
if not entries or entries[0].get('version') != '0.1.1':
    raise RuntimeError('Expected public fluorescence_nuclei_and_cells v0.1.1.')
PUBLIC_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['INSTANSEG_BIOIMAGEIO_PATH'] = str(PUBLIC_MODEL_CACHE_ROOT)
runner = NativeInstanSeg(
    model_type='fluorescence_nuclei_and_cells',
    device=DEVICE, verbosity=0, channels_last=False,
)
network = runner.instanseg.eval()
MODEL_PIXEL_SIZE_UM = float(network.pixel_size)
if abs(MODEL_PIXEL_SIZE_UM - 0.5) > 1e-6:
    raise ValueError(f'Expected public model scale 0.5, got {MODEL_PIXEL_SIZE_UM}.')
print({
    'instanseg_source': str(Path(instanseg.__file__).resolve()),
    'model_pixel_size_um': MODEL_PIXEL_SIZE_UM,
    'cells_and_nuclei': bool(network.cells_and_nuclei),
    'source_pixel_size_um': 0.325,
})

## Read the common image and choose identical crops

Crops are selected using only the shared `R1_DAPI` channel. Each marker set then receives the same spatial crop, but with its own exact channel order.

In [ ]:
SOURCE_PIXEL_SIZE_UM = 0.325
def channel_names_from_ome_xml(xml):
    return re.findall(r'<Channel\b[^>]*Name="([^"]+)"', xml or '')

with tifffile.TiffFile(FULL_MERGE_OME) as tf:
    source_names = channel_names_from_ome_xml(tf.ome_metadata)
    level0 = tf.series[0].levels[0]
    source_shape = tuple(int(v) for v in level0.shape)
    source_channel_to_index = {name: i for i, name in enumerate(source_names)}

all_markers = []
for spec in MARKER_SET_SPECS.values():
    all_markers.extend(spec['channels'])
missing = sorted(set(all_markers) - set(source_channel_to_index))
if missing:
    raise ValueError(f'Missing SLIDE-0330 channels: {missing}')
DAPI_INDEX = source_channel_to_index['R1_DAPI']
DISPLAY_CHANNELS = ['R9_CD68_E3O7V_488', 'R12_CD3E_E4T1B_AF555', 'R1_DAPI']
display_ids = [source_channel_to_index[name] for name in DISPLAY_CHANNELS]
print({'source_shape': source_shape, 'source_channel_count': len(source_names)})

rng = np.random.default_rng(RANDOM_SEED)
source_height, source_width = source_shape[-2:]
if CROP_SIZE_PX >= source_height or CROP_SIZE_PX >= source_width:
    raise ValueError(f'Crop size {CROP_SIZE_PX} does not fit {source_shape[-2:]}')
candidate_count = max(30, N_CROPS * 10)
candidate_coords, candidate_scores = [], []
with tifffile.TiffFile(FULL_MERGE_OME) as tf:
    store = tf.series[0].aszarr(level=0)
    try:
        source = zarr.open(store, mode='r')
        for _ in range(candidate_count):
            x = int(rng.integers(0, source_width - CROP_SIZE_PX))
            y = int(rng.integers(0, source_height - CROP_SIZE_PX))
            dapi = np.asarray(source[DAPI_INDEX, y:y + CROP_SIZE_PX, x:x + CROP_SIZE_PX])
            candidate_coords.append((x, y))
            candidate_scores.append(float(np.percentile(dapi, 99.5)))
        scores = np.asarray(candidate_scores)
        eligible = np.flatnonzero(scores >= np.quantile(scores, 0.5))
        chosen = rng.choice(eligible, size=min(N_CROPS, len(eligible)), replace=False)
        if len(chosen) < N_CROPS:
            raise RuntimeError('Not enough eligible crops were generated.')
        CROP_SPECS = []
        CROP_IMAGES = {name: [] for name in MARKER_SET_SPECS}
        for crop_number, candidate_index in enumerate(chosen, start=1):
            x, y = candidate_coords[int(candidate_index)]
            CROP_SPECS.append({
                'crop': crop_number, 'x': x, 'y': y,
                'width': CROP_SIZE_PX, 'height': CROP_SIZE_PX,
                'dapi_99_5': float(candidate_scores[int(candidate_index)]),
            })
            for set_name, spec in MARKER_SET_SPECS.items():
                channel_ids = [source_channel_to_index[name] for name in spec['channels']]
                crop = np.stack([
                    np.asarray(source[channel_id, y:y + CROP_SIZE_PX, x:x + CROP_SIZE_PX])
                    for channel_id in channel_ids
                ]).astype(np.float32, copy=False)
                CROP_IMAGES[set_name].append(np.ascontiguousarray(crop))
    finally:
        store.close()
display(pd.DataFrame(CROP_SPECS))
display(pd.DataFrame([
    {'marker_set': name, 'label': spec['label'], 'n_channels': len(spec['channels']),
     'channels': spec['channels']}
    for name, spec in MARKER_SET_SPECS.items()
]))
print('Both marker sets use these exact crop coordinates:', [(x['x'], x['y']) for x in CROP_SPECS])

## Run identical public-model inference for both marker sets

In [ ]:
def predict_public(image):
    with torch.inference_mode():
        labels = runner.eval_small_image(
            torch.from_numpy(image),
            pixel_size=SOURCE_PIXEL_SIZE_UM,
            normalise=True,
            return_image_tensor=False,
            target='all_outputs',
            rescale_output=True,
            **POSTPROCESSING,
        )
    labels = labels.squeeze(0).to(torch.int32).cpu().numpy()
    if labels.shape != (2, CROP_SIZE_PX, CROP_SIZE_PX):
        raise ValueError(f'Unexpected output {labels.shape}')
    return labels

PREDICTIONS = {crop_number: {} for crop_number in range(1, N_CROPS + 1)}
started = time.perf_counter()
for crop_number in range(1, N_CROPS + 1):
    print(f'Crop {crop_number}/{N_CROPS}', flush=True)
    for set_name in MARKER_SET_SPECS:
        PREDICTIONS[crop_number][set_name] = predict_public(CROP_IMAGES[set_name][crop_number - 1])
        labels = PREDICTIONS[crop_number][set_name]
        print({
            'marker_set': set_name,
            'nuclei': int(np.unique(labels[0][labels[0] > 0]).size),
            'cells': int(np.unique(labels[1][labels[1] > 0]).size),
        }, flush=True)
print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes.')
del runner, network
gc.collect()
torch.cuda.empty_cache()

## Quantify the difference between the two predictions

The foreground IoU/Dice values below measure agreement between the two marker-set outputs, not accuracy against ground truth. Instance IDs are not expected to match, so the comparison is performed on binary nuclear and cell foreground masks.

In [ ]:
def count_instances(labels):
    return int(np.unique(labels[labels > 0]).size)

def binary_iou(a, b):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 1.0

def binary_dice(a, b):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    denom = a.sum() + b.sum()
    return float(2 * np.logical_and(a, b).sum() / denom) if denom else 1.0

rows = []
for crop_number in range(1, N_CROPS + 1):
    current = PREDICTIONS[crop_number]['current']
    previous = PREDICTIONS[crop_number]['previous_pipeline']
    for target_index, target in enumerate(['nuclei', 'cells']):
        current_fg, previous_fg = current[target_index] > 0, previous[target_index] > 0
        rows.append({
            'crop': crop_number, 'target': target,
            'current_instances': count_instances(current[target_index]),
            'previous_instances': count_instances(previous[target_index]),
            'instance_count_delta': count_instances(current[target_index]) - count_instances(previous[target_index]),
            'foreground_iou': binary_iou(current_fg, previous_fg),
            'foreground_dice': binary_dice(current_fg, previous_fg),
            'disagreement_fraction': float(np.logical_xor(current_fg, previous_fg).mean()),
        })
marker_comparison = pd.DataFrame(rows)
display(marker_comparison)
display(marker_comparison.groupby('target', sort=False)[
    ['current_instances', 'previous_instances', 'instance_count_delta',
     'foreground_iou', 'foreground_dice', 'disagreement_fraction']
].agg({'current_instances': 'mean', 'previous_instances': 'mean',
       'instance_count_delta': 'mean', 'foreground_iou': 'mean',
       'foreground_dice': 'mean', 'disagreement_fraction': 'mean'}))
marker_comparison.to_csv(RESULTS_ROOT / 'marker_set_comparison.csv', index=False)
print('Saved:', RESULTS_ROOT / 'marker_set_comparison.csv')

## Visualize the same crops side by side

Cyan boundaries are nuclei and yellow boundaries are whole cells. The first row shows both compartments; the next two rows isolate nuclei and cells.

In [ ]:
def robust01(image, percentiles=(1.0, 99.8)):
    image = np.asarray(image, dtype=np.float32)
    low, high = np.percentile(image, percentiles)
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - low) / (high - low), 0, 1)

def display_rgb(image, marker_set='current'):
    marker_order = MARKER_SET_SPECS[marker_set]['channels']
    display_indices = [marker_order.index(channel) for channel in DISPLAY_CHANNELS]
    return np.stack([
        robust01(image[display_indices[0]]),
        robust01(image[display_indices[1]]),
        robust01(image[display_indices[2]]),
    ], axis=-1)

def show_prediction(ax, rgb, labels=None, mode='both', title=''):
    ax.imshow(rgb, interpolation='nearest')
    if labels is not None and mode in ('both', 'nuclei'):
        ax.contour(find_boundaries(labels[0], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.45)
    if labels is not None and mode in ('both', 'cells'):
        ax.contour(find_boundaries(labels[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.45)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

for crop_number in range(1, N_CROPS + 1):
    rgb = display_rgb(CROP_IMAGES['current'][crop_number - 1])
    current = PREDICTIONS[crop_number]['current']
    previous = PREDICTIONS[crop_number]['previous_pipeline']
    figure, axes = plt.subplots(3, 3, figsize=(18, 18), squeeze=False)
    for row, mode in enumerate(['both', 'nuclei', 'cells']):
        axes[row, 0].imshow(rgb); axes[row, 0].axis('off'); axes[row, 0].set_title('Input', fontsize=9)
        axes[row, 0].set_ylabel(mode.title(), fontsize=10)
        show_prediction(axes[row, 1], rgb, current, mode=mode, title='Current 11-channel set')
        show_prediction(axes[row, 2], rgb, previous, mode=mode, title='Previous pipeline 9-channel set')
    figure.suptitle(
        f'{SLIDE_ID} crop {crop_number} | resolve={RESOLVE_CELL_AND_NUCLEUS} | '        'cyan=nuclei, yellow=cells', fontsize=13,
    )
    figure.tight_layout()
    if SAVE_FIGURES:
        output_path = RESULTS_ROOT / f'{SLIDE_ID}_crop{crop_number}_marker_sets.png'
        figure.savefig(output_path, dpi=160, bbox_inches='tight')
        print('Saved:', output_path)
    plt.show()

In [ ]:
provenance = {
    'slide_id': SLIDE_ID,
    'source_image': str(FULL_MERGE_OME),
    'source_pixel_size_um': SOURCE_PIXEL_SIZE_UM,
    'model': 'fluorescence_nuclei_and_cells',
    'model_version': '0.1.1',
    'model_pixel_size_um': MODEL_PIXEL_SIZE_UM,
    'marker_sets': {name: spec['channels'] for name, spec in MARKER_SET_SPECS.items()},
    'previous_run_record': str(PREVIOUS_RUN_RECORD),
    'previous_run_record_sha256': hashlib.sha256(PREVIOUS_RUN_RECORD.read_bytes()).hexdigest(),
    'previous_pipeline_instanseg_config': OLD_INSTANSEG_CONFIG,
    'crop_specs': CROP_SPECS,
    'random_seed': RANDOM_SEED,
    'postprocessing_held_fixed': POSTPROCESSING,
    'resolve_cell_and_nucleus': RESOLVE_CELL_AND_NUCLEUS,
    'native_api': 'eval_small_image',
}
(RESULTS_ROOT / 'provenance.json').write_text(json.dumps(provenance, indent=2, default=str) + '\n')
print('Saved:', RESULTS_ROOT / 'provenance.json')